# Attention Mechanisms: From Scaled Dot-Product to Multi-Head Attention

This notebook explores the fundamental building blocks of modern transformer architectures: **attention mechanisms**. We'll build intuition by implementing and visualizing both scaled dot-product attention and multi-head attention from scratch.

**What you'll learn:**
- The core concept of attention: "looking where it matters"
- Query, Key, and Value representations
- Scaled dot-product attention mechanics
- Why scaling matters for stability
- Multi-head attention: learning diverse patterns
- Visualizing attention patterns

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from aiml_notebooks import get_device, set_seed

# Set style and seed for reproducibility
sns.set_style('whitegrid')
set_seed(42)
device = get_device()

## Part 1: The Attention Concept

### The Problem: Not All Context is Equal

When processing sequences (text, time series, etc.), not all elements are equally important for predicting the next element. For example, in the sentence:

> "The cat sat on the mat because **it** was comfortable."

To understand what "it" refers to, we need to **attend** to "cat" or "mat", not equally to all words.

**Attention mechanisms** allow the model to dynamically focus on relevant parts of the input.

### The Core Idea: Query, Key, Value

Attention uses three learned representations:

1. **Query (Q)**: "What am I looking for?" - The current element asking for information
2. **Key (K)**: "What do I contain?" - Descriptions of what each element offers
3. **Value (V)**: "What information do I have?" - The actual information to retrieve

**Analogy**: Searching in a library
- Query = Your search terms
- Keys = Book titles/descriptions  
- Values = Book contents

You compare your query with all keys to find relevant books, then read their contents (values).

### Mathematical Formulation

Given an input sequence of vectors, we:

1. Project to Q, K, V using learned weight matrices
2. Compute attention scores: how much each position should attend to others
3. Use scores to create weighted sum of values

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $QK^T$: Computes similarity between queries and keys
- $\sqrt{d_k}$: Scaling factor (key dimension)
- $\text{softmax}$: Converts scores to probabilities
- Final multiplication with $V$: Weighted sum of values

## Part 2: Scaled Dot-Product Attention

### Step 1: Create Sample Input Sequences

Let's create a simple sequence to work with. We'll use a batch of sequences with:
- Batch size: 2
- Sequence length: 4 
- Embedding dimension: 8

In [ ]:
# Create sample input
batch_size = 2
seq_len = 4
d_model = 8  # embedding dimension

# Random input sequences: (batch, seq_len, d_model)
x = torch.randn(batch_size, seq_len, d_model)
print(f"Input shape: {x.shape}")
print(f"\nFirst sequence (4 tokens, 8-dim embeddings):")
print(x[0])

### Step 2: Create Q, K, V Projections

We'll use linear transformations to project our input into query, key, and value spaces.

In [ ]:
# Dimension for Q, K, V
d_k = d_model  # Often the same as d_model

# Create projection matrices
W_q = nn.Linear(d_model, d_k, bias=False)
W_k = nn.Linear(d_model, d_k, bias=False)
W_v = nn.Linear(d_model, d_k, bias=False)

# Project input to Q, K, V
Q = W_q(x)  # (batch, seq_len, d_k)
K = W_k(x)  # (batch, seq_len, d_k)
V = W_v(x)  # (batch, seq_len, d_k)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

### Step 3: Compute Attention Scores (QK^T)

We calculate how much each query attends to each key using dot products. Higher dot product = more similar = more attention.

In [ ]:
# Compute attention scores: Q * K^T
# (batch, seq_len, d_k) @ (batch, d_k, seq_len) -> (batch, seq_len, seq_len)
scores = torch.matmul(Q, K.transpose(-2, -1))

print(f"Attention scores shape: {scores.shape}")
print(f"\nRaw attention scores for first sequence:")
print(scores[0])
print(f"\nScore range: [{scores.min():.2f}, {scores.max():.2f}]")

### Step 4: Why Scaling Matters

**Problem**: As $d_k$ grows, dot products can become very large, pushing softmax into regions with tiny gradients.

**Intuition**: 
- Dot product of two random vectors grows with dimension
- Large values → softmax concentrates on max → vanishing gradients elsewhere

**Solution**: Scale by $\sqrt{d_k}$ to keep variance constant regardless of dimension.

In [ ]:
# Demonstrate the scaling effect
print(f"d_k = {d_k}")
print(f"sqrt(d_k) = {np.sqrt(d_k):.2f}")

# Scale the scores
scaled_scores = scores / np.sqrt(d_k)

print(f"\nScaled scores range: [{scaled_scores.min():.2f}, {scaled_scores.max():.2f}]")
print(f"\nScaled attention scores for first sequence:")
print(scaled_scores[0])

### Step 5: Apply Softmax to Get Attention Weights

Softmax converts scores to a probability distribution over positions. Each row sums to 1.

In [ ]:
# Apply softmax to get attention weights
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nAttention weights for first sequence:")
print(attention_weights[0])
print(f"\nRow sums (should be 1.0): {attention_weights[0].sum(dim=-1)}")

### Step 6: Weighted Sum of Values

Finally, we use attention weights to compute a weighted sum of the values. This gives us the output for each position.

In [ ]:
# Compute weighted sum: attention_weights @ V
# (batch, seq_len, seq_len) @ (batch, seq_len, d_k) -> (batch, seq_len, d_k)
output = torch.matmul(attention_weights, V)

print(f"Output shape: {output.shape}")
print(f"\nOutput for first sequence:")
print(output[0])

### Visualizing Attention Weights

Let's visualize the attention pattern. Each row shows where that position attends to.

In [ ]:
# Visualize attention weights for first sequence
plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights[0].detach().numpy(), 
            annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=[f'K{i}' for i in range(seq_len)],
            yticklabels=[f'Q{i}' for i in range(seq_len)],
            cbar_kws={'label': 'Attention Weight'})
plt.xlabel('Keys (attending to)')
plt.ylabel('Queries (attending from)')
plt.title('Scaled Dot-Product Attention Weights')
plt.tight_layout()
plt.show()

### Complete Scaled Dot-Product Attention Function

Let's package everything into a clean function.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Queries (batch, seq_len, d_k)
        K: Keys (batch, seq_len, d_k)
        V: Values (batch, seq_len, d_k)
        mask: Optional mask (batch, seq_len, seq_len)
    
    Returns:
        output: Attention output (batch, seq_len, d_k)
        attention_weights: Attention weights (batch, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    
    # Apply mask if provided (e.g., for causal attention)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Apply softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Test the function
output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

## Part 3: Causal (Masked) Attention

### Why Masking?

In language modeling, we predict the next token. Position $i$ should only attend to positions $\leq i$ (not future positions).

We achieve this with a **causal mask**: setting future attention scores to $-\infty$ before softmax.

In [ ]:
# Create causal mask: lower triangular matrix
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

print("Causal mask (1 = attend, 0 = mask):")
print(causal_mask)

### Apply Causal Attention

In [ ]:
# Apply causal attention
causal_output, causal_attn_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print(f"Causal attention weights for first sequence:")
print(causal_attn_weights[0])
print(f"\nNote: Upper triangle is now zero (no attending to future)")

### Visualize Causal Attention

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regular attention
sns.heatmap(attn_weights[0].detach().numpy(), 
            annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=[f'K{i}' for i in range(seq_len)],
            yticklabels=[f'Q{i}' for i in range(seq_len)],
            ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Regular Attention (Bidirectional)')
axes[0].set_xlabel('Keys')
axes[0].set_ylabel('Queries')

# Causal attention
sns.heatmap(causal_attn_weights[0].detach().numpy(), 
            annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=[f'K{i}' for i in range(seq_len)],
            yticklabels=[f'Q{i}' for i in range(seq_len)],
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Causal Attention (No Future Info)')
axes[1].set_xlabel('Keys')
axes[1].set_ylabel('Queries')

plt.tight_layout()
plt.show()

## Part 4: Multi-Head Attention

### The Limitation of Single-Head Attention

Single attention can only learn one type of relationship. But sequences have multiple types of dependencies:
- Syntactic relationships (subject-verb agreement)
- Semantic relationships (word meanings)
- Positional relationships (nearby words)

**Multi-head attention** runs multiple attention mechanisms in parallel, each learning different patterns.

### How Multi-Head Attention Works

1. **Split**: Project input to $h$ different Q, K, V representations ("heads")
2. **Parallel attention**: Apply scaled dot-product attention to each head independently  
3. **Concat**: Concatenate all head outputs
4. **Project**: Linear transformation to combine information

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

where:
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

### Implement Multi-Head Attention

We'll create a proper multi-head attention module.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        Multi-head attention module.
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Linear projections for Q, K, V (all heads at once)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x, mask=None):
        """
        Args:
            x: Input (batch, seq_len, d_model)
            mask: Optional mask (batch, seq_len, seq_len) or (seq_len, seq_len)
        
        Returns:
            output: (batch, seq_len, d_model)
            attention_weights: (batch, num_heads, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.size()
        
        # Linear projections: (batch, seq_len, d_model)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape to (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # Weighted sum of values
        attn_output = torch.matmul(attention_weights, V)
        
        # Concatenate heads: (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # Final linear projection
        output = self.W_o(attn_output)
        
        return output, attention_weights

print("MultiHeadAttention module created!")

### Test Multi-Head Attention

Let's create a multi-head attention module with 4 heads and test it.

In [ ]:
# Create multi-head attention with 4 heads
num_heads = 4
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# Forward pass
mha_output, mha_weights = mha(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {mha_output.shape}")
print(f"Attention weights shape: {mha_weights.shape}")
print(f"  (batch={batch_size}, heads={num_heads}, seq_len={seq_len}, seq_len={seq_len})")

### Visualize Multi-Head Attention Patterns

Each head learns different attention patterns. Let's visualize all 4 heads.

In [ ]:
# Visualize all heads for first sequence
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for head_idx in range(num_heads):
    head_weights = mha_weights[0, head_idx].detach().numpy()
    
    sns.heatmap(head_weights, 
                annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=[f'K{i}' for i in range(seq_len)],
                yticklabels=[f'Q{i}' for i in range(seq_len)],
                ax=axes[head_idx], vmin=0, vmax=1,
                cbar_kws={'label': 'Weight'})
    axes[head_idx].set_title(f'Head {head_idx + 1}')
    axes[head_idx].set_xlabel('Keys')
    axes[head_idx].set_ylabel('Queries')

plt.suptitle('Multi-Head Attention Patterns (4 Heads)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print("Notice: Each head learns different attention patterns!")

## Part 5: Practical Example with Meaningful Sequences

### Create a Sequence with Positional Information

Let's create a more interpretable example where we can see attention patterns emerge.

In [ ]:
# Create a sequence with position-dependent patterns
seq_len = 8
d_model = 16
num_heads = 4

# Create input with positional encoding
positions = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

pos_encoding = torch.zeros(seq_len, d_model)
pos_encoding[:, 0::2] = torch.sin(positions * div_term)
pos_encoding[:, 1::2] = torch.cos(positions * div_term)

# Add batch dimension
x_pos = pos_encoding.unsqueeze(0)

print(f"Sequence with positional encoding shape: {x_pos.shape}")

### Apply Multi-Head Attention

In [ ]:
# Create and apply multi-head attention
mha_large = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
output_pos, weights_pos = mha_large(x_pos)

print(f"Output shape: {output_pos.shape}")
print(f"Attention weights shape: {weights_pos.shape}")

### Visualize Attention Across All Heads

In [ ]:
# Visualize all heads
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for head_idx in range(num_heads):
    head_weights = weights_pos[0, head_idx].detach().numpy()
    
    sns.heatmap(head_weights, 
                annot=True, fmt='.2f', cmap='RdYlBu_r',
                xticklabels=[f'Pos{i}' for i in range(seq_len)],
                yticklabels=[f'Pos{i}' for i in range(seq_len)],
                ax=axes[head_idx], vmin=0, vmax=0.5,
                cbar_kws={'label': 'Attention'})
    axes[head_idx].set_title(f'Head {head_idx + 1}', fontsize=12, fontweight='bold')
    axes[head_idx].set_xlabel('Attending to (Keys)', fontsize=10)
    axes[head_idx].set_ylabel('Attending from (Queries)', fontsize=10)

plt.suptitle('Multi-Head Attention on Positional Encodings', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### Average Attention Across Heads

We can also look at the average attention pattern to see overall tendencies.

In [ ]:
# Average attention across all heads
avg_attention = weights_pos[0].mean(dim=0).detach().numpy()

plt.figure(figsize=(10, 8))
sns.heatmap(avg_attention, 
            annot=True, fmt='.3f', cmap='RdYlBu_r',
            xticklabels=[f'Pos{i}' for i in range(seq_len)],
            yticklabels=[f'Pos{i}' for i in range(seq_len)],
            cbar_kws={'label': 'Average Attention'})
plt.xlabel('Attending to (Keys)', fontsize=11)
plt.ylabel('Attending from (Queries)', fontsize=11)
plt.title('Average Attention Across All Heads', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 6: Attention with Causal Masking in Multi-Head

Let's apply causal masking to our multi-head attention for autoregressive modeling.

In [ ]:
# Create causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

# Apply multi-head attention with causal mask
output_causal, weights_causal = mha_large(x_pos, mask=causal_mask)

print(f"Causal masked output shape: {output_causal.shape}")
print(f"Causal masked weights shape: {weights_causal.shape}")

### Visualize Causal Multi-Head Attention

In [ ]:
# Compare regular vs causal attention for one head
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Regular multi-head attention (head 0)
sns.heatmap(weights_pos[0, 0].detach().numpy(), 
            annot=True, fmt='.2f', cmap='RdYlBu_r',
            xticklabels=[f'Pos{i}' for i in range(seq_len)],
            yticklabels=[f'Pos{i}' for i in range(seq_len)],
            ax=axes[0], vmin=0, vmax=0.5,
            cbar_kws={'label': 'Attention'})
axes[0].set_title('Regular Attention (Head 1)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Attending to', fontsize=10)
axes[0].set_ylabel('Attending from', fontsize=10)

# Causal multi-head attention (head 0)
sns.heatmap(weights_causal[0, 0].detach().numpy(), 
            annot=True, fmt='.2f', cmap='RdYlBu_r',
            xticklabels=[f'Pos{i}' for i in range(seq_len)],
            yticklabels=[f'Pos{i}' for i in range(seq_len)],
            ax=axes[1], vmin=0, vmax=0.5,
            cbar_kws={'label': 'Attention'})
axes[1].set_title('Causal Attention (Head 1)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Attending to', fontsize=10)
axes[1].set_ylabel('Attending from', fontsize=10)

plt.suptitle('Regular vs Causal Multi-Head Attention', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### Visualize All Heads with Causal Masking

In [ ]:
# Visualize all causal heads
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for head_idx in range(num_heads):
    head_weights = weights_causal[0, head_idx].detach().numpy()
    
    sns.heatmap(head_weights, 
                annot=True, fmt='.2f', cmap='RdYlBu_r',
                xticklabels=[f'Pos{i}' for i in range(seq_len)],
                yticklabels=[f'Pos{i}' for i in range(seq_len)],
                ax=axes[head_idx], vmin=0, vmax=0.5,
                cbar_kws={'label': 'Attention'})
    axes[head_idx].set_title(f'Head {head_idx + 1} (Causal)', fontsize=12, fontweight='bold')
    axes[head_idx].set_xlabel('Attending to (Keys)', fontsize=10)
    axes[head_idx].set_ylabel('Attending from (Queries)', fontsize=10)

plt.suptitle('Multi-Head Causal Attention (4 Heads)', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("Note: Each position can only attend to itself and previous positions!")

## Summary

### Key Takeaways

1. **Attention Mechanism**: A way for models to dynamically focus on relevant parts of input
   - Query: "What am I looking for?"
   - Key: "What do I contain?"
   - Value: "What information do I have?"

2. **Scaled Dot-Product Attention**:
   - Compute similarity: $QK^T$
   - Scale by $\sqrt{d_k}$ for stability
   - Softmax to get probabilities
   - Weighted sum of values

3. **Causal Masking**: Prevents attending to future positions (essential for autoregressive models)

4. **Multi-Head Attention**:
   - Run multiple attention heads in parallel
   - Each head learns different patterns
   - Concatenate and project outputs
   - Richer representations from diverse attention patterns

5. **Visualization**: Attention weights show interpretable patterns of what the model focuses on

### Next Steps

- Combine with feedforward layers → Transformer blocks
- Add layer normalization and residual connections
- Build complete encoder/decoder architectures
- Train on real language modeling tasks